# 더 나은 행렬 곱셈

출처: [Triton 행렬 곱셈 튜토리얼](https://triton-lang.org/main/getting-started/tutorials/03-matrix-multiplication.html#sphx-glr-getting-started-tutorials-03-matrix-multiplication-py)

행이나 열 하나씩 처리하는 대신 블록 단위로 연산하는 더 나은 행렬 곱셈을 구현해 보겠습니다. 이렇게 하면 각 스레드 블록의 유휴 영역이 줄어듭니다. 행을 묶어 처리해야 최고 수준의 효율을 얻을 수 있으므로 최적의 구현은 아니지만, 그에 한 걸음 더 가까워집니다.

아래 의사 코드의 핵심은 행렬 `A`의 `MxK` 블록과 행렬 `B`의 `KxN` 블록을 불러오는 것입니다. 그런 다음 `K` 차원을 따라 순회하며 모든 행과 열의 곱을 누적한 뒤, 최종 행렬 곱셈 결과를 `C`에 저장합니다.

```
# 병렬로 실행
for m in range(0, M, BLOCK_SIZE_M):
  # 병렬로 실행
  for n in range(0, N, BLOCK_SIZE_N):
    acc = zeros((BLOCK_SIZE_M, BLOCK_SIZE_N), dtype=float32)
    for k in range(0, K, BLOCK_SIZE_K):
      a = A[m : m+BLOCK_SIZE_M, k : k+BLOCK_SIZE_K]
      b = B[k : k+BLOCK_SIZE_K, n : n+BLOCK_SIZE_N]
      acc += dot(a, b)
    C[m : m+BLOCK_SIZE_M, n : n+BLOCK_SIZE_N] = acc
```

실행 함수를 정의하겠습니다. 이 함수는 `(X.shape[0] / BLOCK_SIZE_X, Y.shape[1] / BLOCK_SIZE_Y)` 모양의 2차원 실행 그리드를 사용합니다. 행 그룹을 순회하고 이에 대응하는 열 그룹과 곱해 출력을 생성합니다.

Triton에서 지원하는 최소 블록 크기는 `16`입니다.

In [ ]:
import torch
from torch.testing import assert_close

import triton
import triton.language as tl

def matmul(X, Y):
    x_rows, x_cols = X.shape
    y_rows, y_cols = Y.shape
    output = torch.zeros(x_rows, y_cols, device="cuda") # 출력 행렬
    
    BLOCK_SIZE_X = 16
    BLOCK_SIZE_Y = 16
    BLOCK_SIZE_K = 32
    
    # 행과 열을 순회하는 2차원 그리드 생성
    grid = lambda meta: (triton.cdiv(x_rows, meta["BLOCK_SIZE_X"]), triton.cdiv(y_cols, meta["BLOCK_SIZE_Y"])) # 행과 열을 순회하는 2차원 실행 그리드
    
    matmul_kernel[grid](X, Y, output, x_rows, x_cols, y_rows, y_cols, BLOCK_SIZE_X=BLOCK_SIZE_X, BLOCK_SIZE_Y=BLOCK_SIZE_Y, BLOCK_SIZE_K=BLOCK_SIZE_K)
    
    return output

이제 커널을 정의합니다. 배열을 사용해 개별 행이나 열이 아니라 데이터 블록을 가리키는 포인터를 생성할 수 있습니다. 핵심은 행과 열 포인터의 인덱스를 올바르게 증가시키는 것입니다.

배열로 포인터를 만드는 예시는 다음과 같습니다. `x_ptrs = x_ptr + (x_offsets[:,None] + k_offsets[None,:])`. `x_offsets[:, None]`은 행 인덱스를, `k_offsets[None,:]`는 열 인덱스를 지정합니다.

- `x_offsets[:,None]`의 결과는 `Nx1` 배열입니다.
- `k_offsets[None,:]`의 결과는 `1xM` 배열입니다.

두 배열을 더하면 데이터 블록에 해당하는 메모리 포인터가 담긴 `NxM` 배열이 됩니다.

다음은 `A`의 왼쪽 위 블록을 가져오는 예시입니다.

In [ ]:
# 메모리 블록 주소 지정 예시
# 8x8 행렬 초기화
A = torch.rand((8,8))

# 행 인덱스
# 열 기준 너비가 8이므로 8을 곱함
# 각 행은 이전 행 시작 위치에서 8만큼 떨어져 시작
a = (torch.arange(4) * 8).reshape(-1,1)
# 첫 4개 열의 인덱스
b = torch.arange(4).reshape(1,-1)

# A의 왼쪽 위 모서리 인덱스
a + b

tensor([[ 0,  1,  2,  3],
        [ 8,  9, 10, 11],
        [16, 17, 18, 19],
        [24, 25, 26, 27]])

이제 커널을 작성하겠습니다. `X`와 `Y` 행렬 각각에 프로그램 ID를 사용하며, 각 ID는 행 또는 열의 한 블록을 가리킵니다.

프로그램 ID를 기준으로 `X`에서 행을, `Y`에서 열을 가져옵니다. 이때 각각 `BLOCK_SIZE_X`, `BLOCK_SIZE_Y`만큼 이동합니다.

그런 다음 내적 차원 `K`를 따라 반복합니다. 처리할 블록은 `ceil(X.shape[1] / K)`개입니다.
- `X` 블록에서 첫 `K`개 열과 `Y` 블록에서 첫 `K`개 행을 선택합니다.
- 두 부분을 곱해 누산기에 저장합니다.
- 행 블록 전체와 열 블록 전체의 곱을 구할 때까지 반복합니다.

결과는 출력 배열의 해당 위치에 저장합니다. 한 번에 블록 전체를 선택하므로 각 행과 열의 끝을 넘지 않도록 마스크를 적용해야 합니다.

아래 프로그램은 다음 작업을 수행합니다.
- 올바른 마스크를 적용해 `X`와 `Y` 블록을 불러옵니다.
- 행렬의 내적 차원을 순회하며 부분 블록끼리 곱합니다.
- 결과를 누산기에 더합니다.
- 출력 행렬의 해당 위치에 결과를 씁니다.

In [ ]:
@triton.jit
def matmul_kernel(
    x_ptr,
    y_ptr,
    output_ptr,
    x_rows,
    x_cols,
    y_rows,
    y_cols,
    BLOCK_SIZE_X: tl.constexpr, # 블록당 행 수
    BLOCK_SIZE_Y: tl.constexpr, # 블록당 열 수
    BLOCK_SIZE_K: tl.constexpr # 반복마다 처리할 내적 차원의 크기
):
    # 2차원 그리드의 프로그램 ID
    x_pid = tl.program_id(axis=0)
    y_pid = tl.program_id(axis=1)

    # X와 Y 포인터의 시작 위치
    # 행을 이동할 때는 열의 스트라이드를, 열을 이동할 때는 행의 스트라이드를 고려
    x_row_start = x_pid * BLOCK_SIZE_X * x_cols # X에서 선택할 블록의 시작 위치
    y_col_start = y_pid * BLOCK_SIZE_Y # Y 블록의 시작 위치

    # 행과 열의 오프셋. 행 오프셋에는 행렬의 열 수를 곱해야 함
    x_offsets = x_row_start + tl.arange(0, BLOCK_SIZE_X) * x_cols # 각 행의 시작 인덱스 (x_cols를 곱함)
    y_offsets = y_col_start + tl.arange(0, BLOCK_SIZE_Y) # 각 열의 시작 인덱스 (스트라이드는 1)

    # 행렬의 내적 차원에 대한 k 오프셋
    k_offsets = tl.arange(0, BLOCK_SIZE_K)

    # 각 행에서 0부터 k까지의 열을 가리키는 X 포인터
    x_ptrs = x_ptr + (x_offsets[:,None] + k_offsets[None,:])

    # 각 열에서 0부터 k까지의 행을 가리키는 Y 포인터
    # 행 시작 위치를 계산하기 위해 k 오프셋에 y_cols를 곱함
    y_ptrs = y_ptr + (k_offsets[:,None] * y_cols + y_offsets[None,:])

    # k를 따라 반복하며 결과를 누적
    # 수치 정밀도를 높이기 위해 float32로 저장
    accumulator = tl.zeros((BLOCK_SIZE_X, BLOCK_SIZE_Y), dtype=tl.float32)
    # BLOCK_SIZE_K씩 이동하며 k 차원 순회
    for k in range(0, tl.cdiv(x_cols, BLOCK_SIZE_K)):
        # 곱할 X 부분 블록 불러오기
        # 각 행의 끝을 넘는 데이터를 불러오지 않도록 마스크 적용
        # [None, :]은 첫 번째 차원에 길이 1을 추가해 마스크가 x_ptrs에 브로드캐스트되도록 함
        a = tl.load(x_ptrs, mask=k_offsets[None, :] < x_cols - k * BLOCK_SIZE_K, other=0.0)

        # [:, None]은 두 번째 차원에 길이 1을 추가해 마스크가 y_ptrs에 브로드캐스트되도록 함
        b = tl.load(y_ptrs, mask=k_offsets[:,None] < y_rows - k * BLOCK_SIZE_K, other=0.0)

        # a와 b를 곱한 뒤 누산기에 더함
        result = tl.dot(a, b)
        accumulator += result

        # 행을 따라 이동하도록 X 포인터 증가
        x_ptrs += BLOCK_SIZE_K

        # 열을 따라 이동하도록 Y 포인터 증가. 행 방향으로 이동하므로 y_cols를 곱함
        y_ptrs += BLOCK_SIZE_K * y_cols

    output = accumulator.to(tl.float16)

    # 출력 포인터 위치 계산
    output_x_start = x_pid * BLOCK_SIZE_X
    output_y_start = y_pid * BLOCK_SIZE_Y

    # 출력 행렬에서 시작할 행 인덱스
    output_x_rows = output_x_start + tl.arange(0, BLOCK_SIZE_X)

    # 출력 오프셋 계산
    output_x_offsets = output_x_start + tl.arange(0, BLOCK_SIZE_X)
    output_y_offsets = output_y_start + tl.arange(0, BLOCK_SIZE_Y)

    # 출력 포인터. 행 인덱스를 메모리 위치로 바꾸기 위해 y_cols를 곱함
    output_ptrs = output_ptr + (output_x_offsets[:, None] * y_cols + output_y_offsets[None, :])

    # 행렬 경계를 벗어나지 않도록 데이터 저장
    # 출력 마스크는 행렬 범위 밖에 쓰지 않도록 보장
    output_mask = (output_x_rows[:, None] < x_rows) & (output_y_offsets[None,:] < y_cols)
    tl.store(output_ptrs, output, mask=output_mask)

이제 커널을 테스트합니다.

In [17]:
torch.manual_seed(0)
x = torch.rand((32, 64), device='cuda', dtype=torch.float16)
y = torch.rand((64, 32), device='cuda', dtype=torch.float16)
output_torch = x @ y
output_triton = matmul(x, y)
assert_close(output_triton, output_torch)
print(output_torch[:5,:5])
print(output_triton[:5,:5])

The maximum difference between torch and triton is 6.995553016662598
tensor([[3.3329, 4.2105, 3.4219, 3.6030, 4.6367, 3.0829, 3.5959, 4.0361, 4.3028,
         4.3413, 4.0192, 3.3523, 4.4383, 4.5726, 3.9396, 5.3161],
        [3.9074, 4.2839, 4.1713, 4.1455, 5.0679, 3.8858, 3.5398, 4.4548, 4.3134,
         4.2419, 4.5389, 3.7586, 4.5478, 4.7638, 3.9674, 5.6322],
        [3.6504, 4.1908, 4.2282, 3.1439, 4.4771, 4.5377, 4.1929, 4.2981, 3.7594,
         4.0305, 4.1960, 3.6005, 4.2225, 4.4191, 3.0318, 4.6951],
        [3.7793, 4.2255, 3.6918, 3.2550, 4.5282, 4.0765, 4.0407, 3.7886, 4.0163,
         4.0619, 4.0634, 3.6067, 4.6882, 4.7348, 3.6913, 5.3068],
        [4.6634, 5.1013, 4.2930, 3.9466, 5.3071, 4.6393, 4.4717, 4.4301, 4.7710,
         4.3534, 5.2003, 3.6226, 4.8308, 5.0126, 3.5417, 6.3912],
        [4.1602, 4.5921, 3.8755, 4.0547, 4.9854, 4.0085, 4.2527, 4.6740, 4.6540,
         4.6436, 4.4359, 4.1250, 5.1606, 4.7144, 3.7987, 5.8863],
        [3.6933, 5.3607, 4.3754, 4.1718, 5.5626, 